# Data Collation: Downloading the SalmoSapro Dataset

This notebook provides the code to download the image dataset used in this study. The image URLs and their corresponding classifications are sourced from the publicly available metadata catalog on Zenodo.

**Reference:**
Olsen, A. S., Cook, N., & Perkins, S. E. (2025). *SalmoSapro Metadata Catalog: A Cross-Platform Index of Salmonid Images with Saprolegnia Classifications* (Version 1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.15097673

### Important Note on Data Availability

As stated in the manuscript, the full dataset used for training consists of both publicly sourced images (downloaded via this notebook) and images provided under specific data sharing agreements.

**Data Availability Statement:**
> The complete code for data processing, model training, and analysis used in this study is publicly available on GitHub at [Link to Your GitHub Repository]. The metadata for all images sourced from public repositories (iNaturalist, Flickr, GBIF, Wikimedia Commons), including the URLs required to download these images, is available on Zenodo (Olsen et al., 2025; DOI: 10.5281/zenodo.15097672). A portion of the ‘Saprolegnia spp.’ class images was provided directly by stakeholders. These images were shared with us for analysis under specific data sharing agreements with the original contributors and are therefore not publicly available for redistribution.

Therefore, running this notebook will download the **publicly available portion** of the dataset.

### Step 1: Download the Metadata

Before running this notebook, you must first download the metadata file from the Zenodo repository.

1. Go to the Zenodo record: [https://zenodo.org/records/15097673](https://zenodo.org/records/15097673)
2. Download the `sapro_final_database.csv` file.
3. Place the downloaded CSV file in the same directory as this notebook.


### Step 2: Setup and Imports

This step imports the necessary Python libraries for this task. You will need `pandas`, `requests`, and `tqdm`. If these are not installed, you can install them by running `pip install pandas requests tqdm` in your terminal after activating your environment.


In [ ]:
import pandas as pd
import requests
from pathlib import Path
import os
from tqdm.auto import tqdm
import time


### Step 3: Configure Paths and Load Metadata

Here, we define where the images will be saved and load the metadata file into a pandas DataFrame.


In [ ]:
# Configuration
METADATA_FILE = 'sapro_final_database.csv'
OUTPUT_DIR = Path('downloaded_dataset')

# Load the metadata
try:
    df = pd.read_csv(METADATA_FILE)
    print(f"Successfully loaded metadata for {len(df)} images.")
    print("Class distribution:")
    print(df['saprolegnia'].value_counts())
except FileNotFoundError:
    print(f"Error: Metadata file '{METADATA_FILE}' not found.")
    print("Please ensure you have downloaded it from Zenodo and placed it in the correct directory.")
    df = None


### Step 4: Download the Images

This is the main part of the notebook. The code below will:
1. Create the necessary output directories (`healthy` and `sapro`).
2. Iterate through each row of the metadata.
3. Attempt to download the image from the specified `image_url`.
4. Save the image to the correct directory based on its `saprolegnia` label.
5. Use the last part of the URL as the filename.

**Note:** This process may take a considerable amount of time depending on your internet connection and the number of images. Some URLs may also be broken or no longer accessible, which is common with web-sourced data.


In [ ]:
def download_images(metadata_df, output_dir):
    if metadata_df is None:
        print("Metadata not loaded. Cannot proceed with download.")
        return

    # Create base output directory
    output_dir.mkdir(exist_ok=True)
    
    # Create class subdirectories
    # Note: Zenodo uses 'yes'/'no', our model folders use 'sapro'/'healthy'
    class_map = {'yes': 'sapro', 'no': 'healthy'}
    for new_dir in class_map.values():
        (output_dir / new_dir).mkdir(exist_ok=True)

    print(f"Images will be saved in: {output_dir.resolve()}\\n")
    
    # Setup tqdm progress bar
    tqdm.pandas(desc="Downloading Images")

    # Use a session for potential connection pooling
    session = requests.Session()
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

    def download_row(row):
        try:
            label = row['saprolegnia']
            url = row['image_url']
            
            # Get the correct class directory name
            class_dir = class_map.get(label.lower())
            if class_dir is None:
                return f"Skipped: Unknown label '{label}'"
            
            # Create a simple filename from the URL
            filename = url.split('/')[-1].split('?')[0]
            if not filename:
                 # Handle cases with no clear filename
                filename = f"{row.name}.jpg"
            
            save_path = output_dir / class_dir / filename

            # Check if file already exists to avoid re-downloading
            if save_path.exists():
                return "Skipped: File already exists"
            
            # Download the image
            response = session.get(url, headers=headers, timeout=15, stream=True)
            response.raise_for_status() # Raise an exception for bad status codes
            
            # Save the image
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            # Be polite to servers
            time.sleep(0.1) 
            return "Success"
        except requests.exceptions.RequestException as e:
            return f"Failed: {e}"
        except Exception as e:
            return f"Failed with general error: {e}"

    # Apply the download function to each row
    results = metadata_df.progress_apply(download_row, axis=1)
    
    print("\\n--- Download Complete ---")
    print("\\nSummary of results:")
    print(results.value_counts())

# Run the download process
download_images(df, OUTPUT_DIR)


### Step 5: Create Taxonomic Subsets

This section contains the code to generate the specific taxonomic subsets used for the analyses in the paper. It uses the `taxonomic_info.csv` file to link the photographs to their respective genera (*Salmo*, *Oncorhynchus*, etc.).

**Prerequisite:** Ensure the `taxonomic_info.csv` file is present in the same directory as this notebook.


In [ ]:
import shutil

def create_subsets(metadata_df, taxo_info_df, base_dataset_dir):
    if metadata_df is None or taxo_info_df is None:
        print("Metadata or taxonomy info not loaded. Cannot create subsets.")
        return

    # --- 1. Robustly merge metadata with taxonomic info to find the Genus ---
    
    # Create a mapping from Species/Genus to the correct Genus name
    species_to_genus = taxo_info_df.dropna(subset=['Species', 'Genus']).set_index('Species')['Genus']
    genus_to_genus = taxo_info_df.dropna(subset=['Genus']).set_index('Genus')['Genus']
    
    def find_genus(taxa_name, s_to_g_map, g_to_g_map):
        if pd.isna(taxa_name):
            return None
        # First, try a direct match on species name
        if taxa_name in s_to_g_map:
            return s_to_g_map[taxa_name]
        # Next, try a direct match on genus name
        if taxa_name in g_to_g_map:
            return g_to_g_map[taxa_name]
        # Finally, if it's a binomial name, try matching the first part as a genus
        parts = taxa_name.split()
        if len(parts) > 1 and parts[0] in g_to_g_map:
            return g_to_g_map[parts[0]]
        return None

    full_df = metadata_df.copy()
    full_df['Genus'] = full_df['taxa'].apply(find_genus, args=(species_to_genus, genus_to_genus))
    
    print(f"Successfully merged metadata with taxonomy. {full_df['Genus'].notna().sum()} of {len(full_df)} rows matched a genus.")

    # --- 2. Define and create dataset subsets ---
    
    SUBSET_BASE_DIR = Path('subsets')
    SUBSET_BASE_DIR.mkdir(exist_ok=True)
    class_map = {'yes': 'sapro', 'no': 'healthy'}

    # Helper function to create symlinks
    def create_symlinks_for_subset(subset_df, subset_name):
        subset_dir = SUBSET_BASE_DIR / subset_name
        print(f"\\nCreating subset: {subset_name} ({len(subset_df)} images)")
        
        # Clean up old directory if it exists
        if subset_dir.exists():
            shutil.rmtree(subset_dir)
        subset_dir.mkdir()
        (subset_dir / 'healthy').mkdir()
        (subset_dir / 'sapro').mkdir()
        
        for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"Creating '{subset_name}'"):
            class_dir = class_map.get(row['saprolegnia'].lower())
            
            # Construct original filename and path
            url = row['image_url']
            filename = url.split('/')[-1].split('?')[0]
            if not filename:
                filename = f"{row.name}.jpg"

            original_path = base_dataset_dir / class_dir / filename
            link_path = subset_dir / class_dir / filename

            if original_path.exists():
                # Create a symlink
                os.symlink(original_path.resolve(), link_path)

    # --- Subset A: All photographs (already exists) ---
    print("Subset 'All photographs' is the base downloaded dataset.")

    # --- Subset B: Taxa with photographs in both classes ---
    class_counts = full_df.groupby('taxa')['saprolegnia'].value_counts().unstack(fill_value=0)
    taxa_in_both = class_counts[(class_counts['yes'] > 0) & (class_counts['no'] > 0)].index
    subset_b_df = full_df[full_df['taxa'].isin(taxa_in_both)]
    create_symlinks_for_subset(subset_b_df, 'B_taxa_in_both_classes')

    # --- Subset C: Taxa with >=10 photographs in both classes ---
    taxa_10_in_both = class_counts[(class_counts['yes'] >= 10) & (class_counts['no'] >= 10)].index
    subset_c_df = full_df[full_df['taxa'].isin(taxa_10_in_both)]
    create_symlinks_for_subset(subset_c_df, 'C_taxa_geq_10_in_both')

    # --- Subset D: Oncorhynchus, >=10 photographs in both classes ---
    onco_df = full_df[full_df['Genus'] == 'Oncorhynchus'].copy()
    onco_class_counts = onco_df.groupby('taxa')['saprolegnia'].value_counts().unstack(fill_value=0)
    onco_taxa_10_in_both = onco_class_counts[(onco_class_counts['yes'] >= 10) & (onco_class_counts['no'] >= 10)].index
    subset_d_df = onco_df[onco_df['taxa'].isin(onco_taxa_10_in_both)]
    create_symlinks_for_subset(subset_d_df, 'D_oncorhynchus_geq_10_in_both')
    
    # --- Subset E: Salmo, >=10 photographs in both classes ---
    salmo_df = full_df[full_df['Genus'] == 'Salmo'].copy()
    salmo_class_counts = salmo_df.groupby('taxa')['saprolegnia'].value_counts().unstack(fill_value=0)
    salmo_taxa_10_in_both = salmo_class_counts[(salmo_class_counts['yes'] >= 10) & (salmo_class_counts['no'] >= 10)].index
    subset_e_df = salmo_df[salmo_df['taxa'].isin(salmo_taxa_10_in_both)]
    create_symlinks_for_subset(subset_e_df, 'E_salmo_geq_10_in_both')

# --- Run the subset creation ---
try:
    taxo_df = pd.read_csv('taxonomic_info.csv')
    create_subsets(df, taxo_df, OUTPUT_DIR)
except FileNotFoundError:
    print("\\nError: 'taxonomic_info.csv' not found. Cannot create subsets.")
    print("Please ensure the file is in the same directory as this notebook.")
except Exception as e:
    print(f"\\nAn error occurred during subset creation: {e}")

